# Baseline Evaluation - 125 Rows per Perturbation (NER Subset)

This notebook evaluates the ContraTICO baseline using **all 125 rows** per perturbation (SEED=42).
It computes **string comparison** (F1, EM, chrF, BLEU) and **SBERT** (cosine similarity) metrics.

Output: `results subset/ner/`

## 1. Environment Setup

In [ ]:
import os
import sys
import subprocess

IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = os.path.exists('/kaggle')

if IN_KAGGLE:
    PROJECT_ROOT = '/kaggle/working/askqe'
    if not os.path.exists(PROJECT_ROOT):
        subprocess.run(['git', 'clone',
                        'https://github.com/Simone280802/AskQE_DNLP_2025-2026.git',
                        PROJECT_ROOT], check=True)
elif IN_COLAB:
    PROJECT_ROOT = '/content/askqe'
    if not os.path.exists(PROJECT_ROOT):
        subprocess.run(['git', 'clone',
                        'https://github.com/Simone280802/AskQE_DNLP_2025-2026.git',
                        PROJECT_ROOT], check=True)
else:
    PROJECT_ROOT = os.getcwd()

print(f'Project root: {PROJECT_ROOT}')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'sentence-transformers', 'bert-score', 'textstat',
                'transformers', 'numpy', 'torch', 'sacrebleu'], check=True)
print('Dependencies installed!')

## 2. Configuration

In [ ]:
import json
import random
import csv
import numpy as np

SEED = 42
SUBSET_SIZE = 125  # Use all rows

RESULTS_BASE = os.path.join(PROJECT_ROOT, 'results Qwen3B baseline')
QA_DIR = os.path.join(RESULTS_BASE, 'contratico', 'baseline', 'QA')
OUTPUT_DIR = os.path.join(RESULTS_BASE, 'contratico', 'baseline', 'results subset', 'ner')

LANGUAGES = ['es', 'fr', 'hi', 'tl', 'zh']
PIPELINES = ['atomic', 'semantic', 'vanilla']
PERTURBATIONS = ['synonym', 'word_order', 'spelling', 'expansion_noimpact',
                 'intensifier', 'expansion_impact', 'omission', 'alteration']

print(f'QA directory: {QA_DIR}')
print(f'Output directory: {OUTPUT_DIR}')
print(f'Subset size: {SUBSET_SIZE} rows per perturbation (all rows)')
print(f'Seed: {SEED}')

## 3. Utility Functions

In [ ]:
import re
import string
from collections import Counter
import sacrebleu

def normalize_answer(s):
    """Lower text and remove punctuation, articles and extra whitespace."""
    def remove_articles(text):
        return re.sub(r'\\b(a|an|the)\\b', ' ', text)
    def white_space_fix(text):
        return ' '.join(text.split())
    def remove_punc(text):
        exclude = set(string.punctuation)
        return ''.join(ch for ch in text if ch not in exclude)
    return white_space_fix(remove_articles(remove_punc(s)))

def f1_score(prediction, ground_truth, normalize=True):
    if normalize:
        pred_tokens = normalize_answer(prediction).split()
        gt_tokens = normalize_answer(ground_truth).split()
    else:
        pred_tokens = prediction.split()
        gt_tokens = ground_truth.split()
    common = Counter(pred_tokens) & Counter(gt_tokens)
    num_same = sum(common.values())
    if num_same == 0:
        return 0
    precision = 1.0 * num_same / len(pred_tokens)
    recall = 1.0 * num_same / len(gt_tokens)
    return (2 * precision * recall) / (precision + recall)

def exact_match_score(prediction, ground_truth, normalize=True):
    if normalize:
        return normalize_answer(prediction) == normalize_answer(ground_truth)
    return prediction == ground_truth

def chrf_score(prediction, ground_truth, normalize=True):
    if normalize:
        return sacrebleu.sentence_chrf(normalize_answer(prediction), [normalize_answer(ground_truth)]).score
    return sacrebleu.sentence_chrf(prediction, [ground_truth]).score

def bleu_score(prediction, ground_truth, normalize=True):
    if normalize:
        return sacrebleu.sentence_bleu(normalize_answer(prediction), [normalize_answer(ground_truth)]).score
    return sacrebleu.sentence_bleu(prediction, [ground_truth]).score

def compare_answers(prediction, ground_truth, normalize=True):
    return (
        f1_score(prediction, ground_truth, normalize),
        exact_match_score(prediction, ground_truth, normalize),
        chrf_score(prediction, ground_truth, normalize),
        bleu_score(prediction, ground_truth, normalize)
    )

def load_jsonl(filepath):
    rows = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

def subsample(source_rows, bt_rows, n, seed):
    """Subsample n rows from source and bt (keeping same indices)."""
    assert len(source_rows) == len(bt_rows), f'Mismatch: {len(source_rows)} vs {len(bt_rows)}'
    if n >= len(source_rows):
        return source_rows, bt_rows
    random.seed(seed)
    indices = sorted(random.sample(range(len(source_rows)), n))
    return [source_rows[i] for i in indices], [bt_rows[i] for i in indices]

def parse_answers(answers):
    """Parse answers field, handling both list and string formats."""
    if isinstance(answers, str):
        try:
            answers = json.loads(answers)
        except json.JSONDecodeError:
            return None
    if not isinstance(answers, list):
        return None
    return answers

print('Utility functions loaded.')

## 4. String Comparison Evaluation

In [ ]:
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

string_results = []

for lang in LANGUAGES:
    for pipeline in PIPELINES:
        source_file = os.path.join(QA_DIR, 'source', f'en-{pipeline}.jsonl')
        source_rows = load_jsonl(source_file)

        for perturbation in PERTURBATIONS:
            bt_file = os.path.join(QA_DIR, 'bt', lang, pipeline, f'{lang}-{pipeline}-{perturbation}.jsonl')

            if not os.path.exists(bt_file):
                print(f'WARNING: Missing {bt_file}')
                continue

            bt_rows = load_jsonl(bt_file)
            src_sub, bt_sub = subsample(source_rows, bt_rows, SUBSET_SIZE, SEED)

            f1_scores, em_scores, chrf_scores, bleu_scores = [], [], [], []
            detail_rows = []

            for src_row, bt_row in zip(src_sub, bt_sub):
                ref_answers = parse_answers(src_row.get('answers', []))
                pred_answers = parse_answers(bt_row.get('answers', []))

                if ref_answers is None or pred_answers is None:
                    continue
                if not ref_answers or not pred_answers or len(ref_answers) != len(pred_answers):
                    continue

                row_scores = []
                for pred, ref in zip(pred_answers, ref_answers):
                    if not isinstance(pred, str):
                        pred = str(pred) if pred is not None else ''
                    if not isinstance(ref, str):
                        ref = str(ref) if ref is not None else ''
                    if not pred.strip() or not ref.strip():
                        continue

                    f1, em, chrf, bleu = compare_answers(pred, ref)
                    f1_scores.append(f1)
                    em_scores.append(em)
                    chrf_scores.append(chrf)
                    bleu_scores.append(bleu)
                    row_scores.append({'f1': f1, 'em': em, 'chrf': chrf, 'bleu': bleu})

                detail_rows.append({
                    'id': bt_row.get('id', 'unknown'),
                    'scores': row_scores
                })

            # Save detailed JSONL
            detail_dir = os.path.join(OUTPUT_DIR, 'evaluation', 'string_comparison', pipeline)
            os.makedirs(detail_dir, exist_ok=True)
            detail_file = os.path.join(detail_dir, f'string_comparison_{lang}_{perturbation}.jsonl')
            with open(detail_file, 'w', encoding='utf-8') as f:
                for row in detail_rows:
                    f.write(json.dumps(row, ensure_ascii=False) + '\n')

            # Aggregate
            if f1_scores:
                string_results.append({
                    'language': lang,
                    'pipeline': pipeline,
                    'perturbation': perturbation,
                    'f1': round(np.mean(f1_scores), 4),
                    'em': round(np.mean(em_scores), 4),
                    'chrf': round(np.mean(chrf_scores), 4),
                    'bleu': round(np.mean(bleu_scores), 4),
                    'num_comparisons': len(f1_scores)
                })

            print(f'  {lang}/{pipeline}/{perturbation}: {len(f1_scores)} comparisons')

# Save string comparison summary CSV
sc_csv = os.path.join(OUTPUT_DIR, 'evaluation', 'string_comparison', 'string_comparison_summary.csv')
os.makedirs(os.path.dirname(sc_csv), exist_ok=True)
with open(sc_csv, 'w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=['language', 'pipeline', 'perturbation', 'f1', 'em', 'chrf', 'bleu', 'num_comparisons'])
    w.writeheader()
    w.writerows(string_results)

print(f'\nString comparison summary saved: {sc_csv}')
print(f'Total entries: {len(string_results)}')

## 5. SBERT Evaluation

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel
import torch.nn.functional as F

def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0]
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

sbert_tokenizer = AutoTokenizer.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')
sbert_model = AutoModel.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
sbert_model = sbert_model.to(device)
print(f'SBERT model loaded on {device}')

In [ ]:
sbert_results = []

for lang in LANGUAGES:
    for pipeline in PIPELINES:
        source_file = os.path.join(QA_DIR, 'source', f'en-{pipeline}.jsonl')
        source_rows = load_jsonl(source_file)

        for perturbation in PERTURBATIONS:
            bt_file = os.path.join(QA_DIR, 'bt', lang, pipeline, f'{lang}-{pipeline}-{perturbation}.jsonl')

            if not os.path.exists(bt_file):
                print(f'WARNING: Missing {bt_file}')
                continue

            bt_rows = load_jsonl(bt_file)
            src_sub, bt_sub = subsample(source_rows, bt_rows, SUBSET_SIZE, SEED)

            cos_sims = []

            for src_row, bt_row in zip(src_sub, bt_sub):
                ref_answers = parse_answers(src_row.get('answers', []))
                pred_answers = parse_answers(bt_row.get('answers', []))

                if ref_answers is None or pred_answers is None:
                    continue
                if not ref_answers or not pred_answers or len(ref_answers) != len(pred_answers):
                    continue

                for pred, ref in zip(pred_answers, ref_answers):
                    if not isinstance(pred, str) or not isinstance(ref, str):
                        continue
                    if not pred.strip() or not ref.strip():
                        continue

                    encoded_pred = sbert_tokenizer(pred, padding=True, truncation=True, return_tensors='pt').to(device)
                    encoded_ref = sbert_tokenizer(ref, padding=True, truncation=True, return_tensors='pt').to(device)

                    with torch.no_grad():
                        pred_output = sbert_model(**encoded_pred)
                        ref_output = sbert_model(**encoded_ref)

                    pred_embed = F.normalize(mean_pooling(pred_output, encoded_pred['attention_mask']), p=2, dim=1)
                    ref_embed = F.normalize(mean_pooling(ref_output, encoded_ref['attention_mask']), p=2, dim=1)

                    cos_sim = F.cosine_similarity(pred_embed, ref_embed, dim=1).mean().item()
                    cos_sims.append(cos_sim)

            if cos_sims:
                sbert_results.append({
                    'language': lang,
                    'pipeline': pipeline,
                    'perturbation': perturbation,
                    'cosine_similarity': round(np.mean(cos_sims), 4),
                    'num_comparisons': len(cos_sims)
                })

            print(f'  {lang}/{pipeline}/{perturbation}: {len(cos_sims)} comparisons, avg={np.mean(cos_sims):.4f}' if cos_sims else f'  {lang}/{pipeline}/{perturbation}: no comparisons')

# Save SBERT summary CSV
sbert_csv = os.path.join(OUTPUT_DIR, 'evaluation', 'sbert', 'sbert_summary.csv')
os.makedirs(os.path.dirname(sbert_csv), exist_ok=True)
with open(sbert_csv, 'w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=['language', 'pipeline', 'perturbation', 'cosine_similarity', 'num_comparisons'])
    w.writeheader()
    w.writerows(sbert_results)

print(f'\nSBERT summary saved: {sbert_csv}')
print(f'Total entries: {len(sbert_results)}')

## 6. Combined Summary

In [ ]:
# Merge string comparison and SBERT results into one summary CSV
sbert_lookup = {}
for r in sbert_results:
    key = (r['language'], r['pipeline'], r['perturbation'])
    sbert_lookup[key] = r['cosine_similarity']

combined = []
for r in string_results:
    key = (r['language'], r['pipeline'], r['perturbation'])
    combined.append({
        'language': r['language'],
        'pipeline': r['pipeline'],
        'perturbation': r['perturbation'],
        'f1': r['f1'],
        'em': r['em'],
        'chrf': r['chrf'],
        'bleu': r['bleu'],
        'sbert': sbert_lookup.get(key, ''),
        'num_comparisons': r['num_comparisons']
    })

summary_csv = os.path.join(OUTPUT_DIR, 'summary_metrics.csv')
with open(summary_csv, 'w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=['language', 'pipeline', 'perturbation', 'f1', 'em', 'chrf', 'bleu', 'sbert', 'num_comparisons'])
    w.writeheader()
    w.writerows(combined)

print(f'Combined summary saved: {summary_csv}')
print(f'Total entries: {len(combined)}')
print('\nDone!')